In [61]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/nikhil7280/student-performance-multiple-linear-regression/Student_Performance.csv


In [62]:
df = pd.read_csv("/kaggle/input/datasets/nikhil7280/student-performance-multiple-linear-regression/Student_Performance.csv")
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,Yes,9,1,91.0
1,4,82,No,4,2,65.0
2,8,51,Yes,7,2,45.0
3,5,52,Yes,5,2,36.0
4,7,75,No,8,5,66.0


In [63]:
unique_val = df['Extracurricular Activities'].unique()
print(unique_val)

['Yes' 'No']


In [64]:
df['Extracurricular Activities'] = df['Extracurricular Activities'].map({'Yes':1,'No':0})
df.head()

,Hours Studied,Previous Scores,Extracurricular Activities,Sleep Hours,Sample Question Papers Practiced,Performance Index
0,7,99,1,9,1,91.0
1,4,82,0,4,2,65.0
2,8,51,1,7,2,45.0
3,5,52,1,5,2,36.0
4,7,75,0,8,5,66.0


In [65]:
X, y = df.drop(columns=['Performance Index']), df['Performance Index']
print(X[:2])
print(y[:2])

   Hours Studied  Previous Scores  Extracurricular Activities  Sleep Hours  \
0              7               99                           1            9   
1              4               82                           0            4   

   Sample Question Papers Practiced  
0                                 1  
1                                 2  
0    91.0
1    65.0
Name: Performance Index, dtype: float64


In [66]:
X_data = np.array(X)
y_data = np.array(y).reshape(-1,1)
print(X_data[:2])
print(y_data[:2])

[[ 7 99  1  9  1]
 [ 4 82  0  4  2]]
[[91.]
 [65.]]


In [67]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

X_train_data, X_test_data, y_train_data, y_test_data = train_test_split(X_data, y_data, test_size = 0.2, random_state = 42)

X_scaler = StandardScaler()
y_scaler = StandardScaler() 

X_train_scaler = X_scaler.fit_transform(X_train_data)
X_test_scaler = X_scaler.transform(X_test_data)

y_train_scaler = y_scaler.fit_transform(y_train_data)
y_test_scaler = y_scaler.transform(y_test_data)

print(X_train_scaler[:2])
print(y_train_scaler[:2])


[[ 0.00645547 -1.18384439 -0.98930717  0.26988848  0.13404112]
 [-1.14974745 -1.24150571  1.01080841  0.26988848  0.48356164]]
[[-1.00576232]
 [-1.57865337]]


In [68]:
import torch

X_train_scaled_torch = torch.tensor(X_train_scaler, dtype = torch.float32)
X_test_scaled_torch = torch.tensor(X_test_scaler, dtype = torch.float32)

y_train_scaled_torch = torch.tensor(y_train_scaler, dtype = torch.float32)
y_test_scaled_torch = torch.tensor(y_test_scaler, dtype = torch.float32)


print(X_train_scaled_torch[:2])
print(y_train_scaled_torch[:2])

tensor([[ 0.0065, -1.1838, -0.9893,  0.2699,  0.1340],
        [-1.1497, -1.2415,  1.0108,  0.2699,  0.4836]])
tensor([[-1.0058],
        [-1.5787]])


In [69]:
import torch.nn as nn
import torch.optim as optim 

input_size = X_data.shape[1]
model = nn.Linear(input_size,1)

loss_fun = nn.MSELoss()
optimizer = optim.SGD(model.parameters(),lr=0.01)

epochs = 1000 
for epoch in range(epochs):
    model.train()
    train_output = model(X_train_scaled_torch)
    loss = loss_fun(train_output,y_train_scaled_torch)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if(epoch % 200 == 0 ):
        print(loss.item())


from sklearn.metrics import r2_score,mean_absolute_error
model.eval()
with torch.no_grad():
    test_output = model(X_test_scaled_torch)
    r2 = r2_score(y_test_scaled_torch,test_output)
    mae = mean_absolute_error(y_test_scaled_torch,test_output)
    print(f"r2_score {r2*100}")
    print(f"mean_sbsolute_error {mae*10000}")

1.8269100189208984
0.011934476904571056
0.011310338973999023
0.011310121044516563
0.011310119181871414
r2_score 98.89832725662629
mean_sbsolute_error 839.0888160665054


In [70]:
X = [[6, 99,  1 , 9 , 1]]

model.eval()
with torch.no_grad():
    X_data_scaled = X_scaler.transform(X)
    X_scaled_torch = torch.tensor(X_data_scaled, dtype = torch.float32)
    output_data = model(X_scaled_torch)
    output_normal = y_scaler.inverse_transform(output_data)
    print(output_normal)

[[88.96765954]]
